In [ ]:
!pip install transformers torch accelerate bitsandbytes soundfile librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00


In [ ]:
import os
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

# =====================================================================
# CONFIGURATION: Set your uploaded audio filename here
# =====================================================================
YOUR_AUDIO_FILE = "/content/Full Narration_MarauliKhurad.m4a"

print("🚀 Launching Sarvam-30B Local Audio Pipeline...")

if not os.path.exists(YOUR_AUDIO_FILE):
    print(f"❌ File Not Found: '{YOUR_AUDIO_FILE}'")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🖥️ Target Computing Unit: {device.upper()}")

    try:
        # --- STAGE 1: Audio Transcription Engine (Whisper-Turbo) ---
        print("\n📥 Loading localized acoustic layer (Whisper-Turbo)...")
        audio_decoder = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-large-v3-turbo",
            device=device,
            chunk_length_s=30
        )

        print("🔄 Extracting Punjabi natively into Gurmukhi Script...")
        audio_result = audio_decoder(
            YOUR_AUDIO_FILE,
            return_timestamps=True,
            generate_kwargs={"language": "pa", "task": "transcribe"}
        )
        punjabi_transcript = audio_result["text"].strip()
        print("   ✅ Gurmukhi Raw Transcript Generated Successfully.")

        # --- STAGE 2: The Hero Layer (Local Sarvam-30B Script Mapping) ---
        print("\n📥 Downloading Sarvam-30B-FP8 Weights into VRAM...")
        sarvam_model_id = "sarvamai/sarvam-30b-fp8"

        sarvam_tokenizer = AutoTokenizer.from_pretrained(sarvam_model_id)
        sarvam_model = AutoModelForCausalLM.from_pretrained(
            sarvam_model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )

        # Few-Shot Instruction Prompt to lock down the 30B model behavior
        sarvam_prompt = (
            f"Task: Convert the following Punjabi text from Gurmukhi script to Devanagari (Hindi) script phonetically. "
            f"Do not translate the meaning. Keep the exact Punjabi words, just change the alphabet letters.\n\n"
            f"Example 1:\n"
            f"Gurmukhi: ਮੈਂ ਕੀ ਕਰਾਂ?\n"
            f"Devanagari: मैं की करां?\n\n"
            f"Current Task:\n"
            f"Gurmukhi: {punjabi_transcript}\n"
            f"Devanagari:"
        )

        print("🧠 Sarvam-30B executing script-swap layout mapping locally...")
        inputs = sarvam_tokenizer(sarvam_prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_tokens = sarvam_model.generate(
                **inputs,
                max_new_tokens=1024,
                temperature=0.01, # Keep creativity minimal to block text translations
                do_sample=False
            )

        decoded_output = sarvam_tokenizer.decode(output_tokens[0], skip_special_tokens=True)

        # Isolate Sarvam's final script generation
        hindi_transcript = decoded_output.split("Current Task:\n")[-1].split("Devanagari:")[-1].strip()
        print("   ✅ Sarvam-30B processing completed successfully.")

        # --- STAGE 3: Export Clean Deliverable Files ---
        punjabi_out_file = "/content/transcript_punjabi.txt"
        hindi_out_file = "/content/transcript_hindi.txt"

        with open(punjabi_out_file, "w", encoding="utf-8") as f:
            f.write(punjabi_transcript)

        with open(hindi_out_file, "w", encoding="utf-8") as f:
            f.write(hindi_transcript)

        print("\n✨ PIPELINE RUN SUCCESSFUL!")
        print(f"💾 File 1 (Pure Punjabi Words in Gurmukhi Script) -> {punjabi_out_file}")
        print(f"💾 File 2 (Pure Punjabi Words in Devanagari Script via 30B) -> {hindi_out_file}")
        print("\n👉 Click Refresh on the sidebar folder tree to grab your files!")

    except Exception as e:
        print(f"\n❌ Local Execution Error: {str(e)}")

🚀 Launching Sarvam-30B Local Audio Pipeline...
🖥️ Target Computing Unit: CPU

📥 Loading localized acoustic layer (Whisper-Turbo)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


🔄 Extracting Punjabi natively into Gurmukhi Script...


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

   ✅ Gurmukhi Raw Transcript Generated Successfully.

📥 Downloading Sarvam-30B-FP8 Weights into VRAM...


config.json:   0%|          | 0.00/2.42k [00:00<?, ?B/s]